# Section 1: Setup and Databset Init
Import SparkSession from pyspark.sql, and initialize a SparkSession. Also import all the necessary functions from pyspark.sql.functions and all necessary classes from pyspark.sql.types

In [1]:
from pyspark.sql.functions import broadcast, explode, split, col, avg, round, desc
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

In [2]:
# Create a SparkSession called 'spark'
spark = SparkSession.builder \
    .appName("Week 6 Project 1 Works")\
    .master("local[*]") \
    .getOrCreate()

sc = spark.sparkContext
sc.setLogLevel("ERROR")  # Suppress warnings for cleaner output

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/02/23 16:16:15 WARN Utils: Your hostname, MacBook-Pro-3.local, resolves to a loopback address: 127.0.0.1; using 192.168.5.102 instead (on interface en0)
26/02/23 16:16:15 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/23 16:16:16 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Users
This is the given schema for the users dataset in the README:

UserID::Gender::Age::Occupation::Zip-code


In [3]:
schema_users = StructType([
    StructField("UserID", IntegerType(), True),
    StructField("Gender", StringType(), True),
    StructField("Age", IntegerType(), True),
    StructField("Occupation",IntegerType(), True),
    StructField("Zip-code", StringType(), True)
])

df_users = spark.read \
    .option("header", "false") \
    .option("quote", '"') \
    .option("escape", '"') \
    .option("multiLine", "true") \
    .option("sep", "::") \
    .schema(schema_users) \
    .csv("./ml-1m/users.dat")

Print the schema of the users table:

In [4]:
df_users.printSchema()

root
 |-- UserID: integer (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Occupation: integer (nullable = true)
 |-- Zip-code: string (nullable = true)



How many entries in the users table?

In [5]:
df_users.count()

6040

Show a few rows from the users table:

In [6]:
df_users.show(5)

+------+------+---+----------+--------+
|UserID|Gender|Age|Occupation|Zip-code|
+------+------+---+----------+--------+
|     1|     F|  1|        10|   48067|
|     2|     M| 56|        16|   70072|
|     3|     M| 25|        15|   55117|
|     4|     M| 45|         7|   02460|
|     5|     M| 25|        20|   55455|
+------+------+---+----------+--------+
only showing top 5 rows


## Ratings 
This is the given schema for the ratings dataset in the README:

UserID::MovieID::Rating::Timestamp

In [7]:
schema_ratings  = StructType([
    StructField("UserID", IntegerType(), True),
    StructField("MovieID", IntegerType(), True),
    StructField("Rating", IntegerType(), True),
    StructField("Timestamp",IntegerType(), True)
])

df_ratings = spark.read \
    .option("header", "false") \
    .option("quote", '"') \
    .option("escape", '"') \
    .option("multiLine", "true") \
    .option("sep", "::") \
    .schema(schema_ratings) \
    .csv("./ml-1m/ratings.dat")

df_ratings.show(5)

+------+-------+------+---------+
|UserID|MovieID|Rating|Timestamp|
+------+-------+------+---------+
|     1|   1193|     5|978300760|
|     1|    661|     3|978302109|
|     1|    914|     3|978301968|
|     1|   3408|     4|978300275|
|     1|   2355|     5|978824291|
+------+-------+------+---------+
only showing top 5 rows


Print the schema of the ratings table:

In [8]:
df_ratings.printSchema()

root
 |-- UserID: integer (nullable = true)
 |-- MovieID: integer (nullable = true)
 |-- Rating: integer (nullable = true)
 |-- Timestamp: integer (nullable = true)



How many entries in the ratings table?

In [9]:
df_ratings.count()

1000209

Show a few rows from the ratings table:

In [10]:
df_ratings.show(5)

+------+-------+------+---------+
|UserID|MovieID|Rating|Timestamp|
+------+-------+------+---------+
|     1|   1193|     5|978300760|
|     1|    661|     3|978302109|
|     1|    914|     3|978301968|
|     1|   3408|     4|978300275|
|     1|   2355|     5|978824291|
+------+-------+------+---------+
only showing top 5 rows


## Movies 
This is the given schema for the movies dataset in the README:

MovieID::Title::Genres

In [11]:
schema_movies  = StructType([
    StructField("MovieID", IntegerType(), True),
    StructField("Title", StringType(), True),
    StructField("Genres",StringType(), True)
])

df_movies = spark.read \
    .option("header", "false") \
    .option("quote", '"') \
    .option("escape", '"') \
    .option("multiLine", "true") \
    .option("sep", "::") \
    .schema(schema_movies) \
    .csv("./ml-1m/movies.dat")



Print the schema of the movie table:

In [12]:
df_movies.printSchema()

root
 |-- MovieID: integer (nullable = true)
 |-- Title: string (nullable = true)
 |-- Genres: string (nullable = true)



How many entries in the moveis table?

In [13]:
df_movies.count()

3883

Show a few rows from the movies table:

In [14]:
df_movies.show(5)

+-------+--------------------+--------------------+
|MovieID|               Title|              Genres|
+-------+--------------------+--------------------+
|      1|    Toy Story (1995)|Animation|Childre...|
|      2|      Jumanji (1995)|Adventure|Childre...|
|      3|Grumpier Old Men ...|      Comedy|Romance|
|      4|Waiting to Exhale...|        Comedy|Drama|
|      5|Father of the Bri...|              Comedy|
+-------+--------------------+--------------------+
only showing top 5 rows


# Section 2: Join the Tables
Ratings is joined with Movies on MovieID, and joined with Users on UserID, and the result is stored in df_joined.

In [15]:
df_joined = df_ratings \
    .join(broadcast(df_users), on="UserID", how="inner") \
    .join(broadcast(df_movies), on="MovieID", how="inner")

Print the schema of the joined tables:

In [16]:
df_joined.printSchema()

root
 |-- MovieID: integer (nullable = true)
 |-- UserID: integer (nullable = true)
 |-- Rating: integer (nullable = true)
 |-- Timestamp: integer (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Occupation: integer (nullable = true)
 |-- Zip-code: string (nullable = true)
 |-- Title: string (nullable = true)
 |-- Genres: string (nullable = true)



How many columns in the joined table?

In [17]:
len(df_joined.columns)

10

How many entries in the joined table?

In [18]:
df_joined.count()

1000209

Show a few entries from the joined table:

In [19]:
df_joined.show(5)

+-------+------+------+---------+------+---+----------+--------+--------------------+--------------------+
|MovieID|UserID|Rating|Timestamp|Gender|Age|Occupation|Zip-code|               Title|              Genres|
+-------+------+------+---------+------+---+----------+--------+--------------------+--------------------+
|   1193|     1|     5|978300760|     F|  1|        10|   48067|One Flew Over the...|               Drama|
|    661|     1|     3|978302109|     F|  1|        10|   48067|James and the Gia...|Animation|Childre...|
|    914|     1|     3|978301968|     F|  1|        10|   48067| My Fair Lady (1964)|     Musical|Romance|
|   3408|     1|     4|978300275|     F|  1|        10|   48067|Erin Brockovich (...|               Drama|
|   2355|     1|     5|978824291|     F|  1|        10|   48067|Bug's Life, A (1998)|Animation|Childre...|
+-------+------+------+---------+------+---+----------+--------+--------------------+--------------------+
only showing top 5 rows


Explain the join operation:

In [20]:
df_joined.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [MovieID#37, UserID#36, Rating#38, Timestamp#39, Gender#1, Age#2, Occupation#3, Zip-code#4, Title#84, Genres#85]
   +- BroadcastHashJoin [MovieID#37], [MovieID#83], Inner, BuildRight, false
      :- Project [UserID#36, MovieID#37, Rating#38, Timestamp#39, Gender#1, Age#2, Occupation#3, Zip-code#4]
      :  +- BroadcastHashJoin [UserID#36], [UserID#0], Inner, BuildRight, false
      :     :- Filter (isnotnull(UserID#36) AND isnotnull(MovieID#37))
      :     :  +- FileScan csv [UserID#36,MovieID#37,Rating#38,Timestamp#39] Batched: false, DataFilters: [isnotnull(UserID#36), isnotnull(MovieID#37)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/Users/ruts/Projects/cs570_movie-lens_group2/ml-1m/ratings.dat], PartitionFilters: [], PushedFilters: [IsNotNull(UserID), IsNotNull(MovieID)], ReadSchema: struct<UserID:int,MovieID:int,Rating:int,Timestamp:int>
      :     +- BroadcastExchange HashedRelationBroadcastMode(Lis

# Section 3: Basic Statistics

In [21]:
df_joined.describe().show()

+-------+------------------+------------------+------------------+--------------------+-------+------------------+-----------------+-----------------+--------------------+-------+
|summary|           MovieID|            UserID|            Rating|           Timestamp| Gender|               Age|       Occupation|         Zip-code|               Title| Genres|
+-------+------------------+------------------+------------------+--------------------+-------+------------------+-----------------+-----------------+--------------------+-------+
|  count|           1000209|           1000209|           1000209|             1000209|1000209|           1000209|          1000209|          1000209|             1000209|1000209|
|   mean|1865.5398981612843| 3024.512347919285| 3.581564453029317| 9.722436954046655E8|   NULL| 29.73831369243828|8.036138447064564|223239.8917114074|                NULL|   NULL|
| stddev|  1096.04068945728|1728.4126948999556|1.1171018453732602|1.2152558939921338E7|   NULL|11.75

## Observations
- Despite the zipcode being cast as a string rather than an integer, they describe() still calculates a numerical mean and standard deviation. Zipcodes are fixed, location based entries, and have no correlation to each other.
- The Age column is cast as an integer, but it is a categorical variable, as it represents age groups, not distinct ages. At first glance, seeing the minimum age as 1 is odd, but it is a valid entry.
- The Occupation column is cast as an integer, but it is a categorical variable, as it represents occupation groups, not distinct occupations. Executing operations of mean and standard deviation on this column is not valid.

# Section 4: EDA Questions
1. *How many unique genres appear in the dataset? Remember that the Genres column
contains pipe-separated values (e.g., “Action|Comedy”). Count the distinct individual
genres, not the distinct genre combinations.*
2. *What is the average rating given by users in the 25–34 age group? Report to at
least two decimal places.*
3. *Which movie has the most ratings, and how many ratings does it have? Report
both the movie title and the count.*

In [23]:
# --- Question 1: Unique Genres ---
# Split the genres string by pipe and explode into separate rows
df_genres = df_movies.withColumn("Genre", explode(split(col("Genres"), "\\|")))
unique_genres_count = df_genres.select("Genre").distinct().count()
print(f"A. Number of unique genres: {unique_genres_count}")

# --- Question 2: Average rating for 25-34 age group ---
# In MovieLens 1M dataset, Age=25 represents the 25-34 age bracket
avg_rating_row = (
    df_joined.filter(col("Age") == 25)
    .agg(round(avg("Rating"), 2).alias("AvgRating"))
    .collect()[0]
)
print(f"B. Average rating for 25-34 age group: {avg_rating_row['AvgRating']:.2f}")

# --- Question 3: Movie with the most ratings ---
most_rated_movie = df_joined.groupBy("Title").count().orderBy(desc("count")).first()
print(
    f"C. Movie with most ratings: '{most_rated_movie['Title']}' with {most_rated_movie['count']} ratings"
)

A. Number of unique genres: 18
B. Average rating for 25-34 age group: 3.55
C. Movie with most ratings: 'American Beauty (1999)' with 3428 ratings


### Question 1. How many unique genres appear in the dataset?
*Answer*: 18 unique genres. 
*Code implementation*: We exploded the pipe-separated Genres column into separate rows using the explode and split functions, then counted the distinct genres.

### Question 2. What is the average rating given by users in the 25–34 age group?
*Answer*: 3.55. 
*Code implementation*: We filtered df_joined for Age == 25 (which represents the 25-34 age bracket in the MovieLens 1M dataset metadata). We aggregated the Rating column using avg() and formatted it to two decimal places using the round() function.

### Question 3. Which movie has the most ratings, and how many ratings does it have?
*Answer*: 'American Beauty (1999)' with 3428 ratings. 
*Code implementation*: We grouped the joined dataset by Title, counted the number of occurrences for each title (representing ratings), ordered by the count in descending order, and took the top result.

# Section 5: Data Quality Observations

## 1. Missing or Null Values
Real-world datasets often have missing entries because a user didn't provide demographic data (like `Zip-code`) or a movie wasn't categorized into a `Genre`. In PySpark, when schemas are explicitly defined or data is loosely cast, invalid formats might also be silently coerced to `null`.

In [26]:
from pyspark.sql.functions import when, count

# Check for Null values in all columns of df_users
# (Removed isnan() since columns are string or integer)
df_users.select([
    count(when(col(c).isNull(), c)).alias(c) for c in df_users.columns
]).show()

# Additionally, check for empty strings in the Zip-code column specifically
empty_zips = df_users.filter(col("Zip-code") == "").count()
print(f"Empty Zip-code strings: {empty_zips}")



+------+------+---+----------+--------+
|UserID|Gender|Age|Occupation|Zip-code|
+------+------+---+----------+--------+
|     0|     0|  0|         0|       0|
+------+------+---+----------+--------+

Empty Zip-code strings: 0


### How to handle it
- **For Numerical/Demographic Columns (Age/Occupation)**: If the percentage of missing data is small, you can drop the rows using `df_users.dropna()`. If large, impute values using the median or mean value for numerical sets.

- **For String Columns (Genres/Zip-code)**: Best practice is to replace non-existent values with a placeholder category like `"Unknown"` using `df.fillna("Unknown", subset=["Zip-code"])`.

## 2. Out-of-Bound / Invalid Ratings
Description: the Rating column should be strictly bound between a predefined scale (1 to 5 stars for the MovieLens dataset). Any rating below 1 or above 5 indicates corrupted data or a data entry anomaly that could skew averages and recommendation algorithms.

In [27]:
# Check for ratings strictly outside the valid 1 to 5 integer range
invalid_ratings = df_ratings.filter((col("Rating") < 1) | (col("Rating") > 5))

invalid_count = invalid_ratings.count()
print(f"Number of invalid outlier ratings: {invalid_count}")

# Option to inspect the corrupted data:
if invalid_count > 0:
    invalid_ratings.show(5)

Number of invalid outlier ratings: 0


### How to handle it
- If the invalid ratings are just a small handful of stray values from faulty telemetry/scraping, the safest method is dropping them entirely:
`df_ratings = df_ratings.filter((col("Rating") >= 1) & (col("Rating") <= 5))`

- Alternatively, if we notice a systematic issue (e.g., someone inputted a 0 instead of a 1, or ratings are coming in on a 1-10 scale), we would use withColumn() to apply a mathematical transformation or when().otherwise() clamping logic to fix them programmatically.